In [1]:
import pandas as pd
from pathlib import Path

In [2]:
# File sumber
INPUT_FILE = Path("retailrocket/events.csv")

# Folder dan file hasil filter
OUTPUT_DIR = Path("data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

P1_FILE = OUTPUT_DIR / "events_P1_training.csv"
P2_FILE = OUTPUT_DIR / "events_P2_testing.csv"

CHUNK_SIZE = 200_000

# Batas waktu UTC:
# Tanggal akhir menggunakan batas eksklusif agar seluruh hari tercakup.
P1_START = int(pd.Timestamp("2015-07-01", tz="UTC").timestamp() * 1000)
P1_END   = int(pd.Timestamp("2015-07-16", tz="UTC").timestamp() * 1000)

P2_START = int(pd.Timestamp("2015-09-01", tz="UTC").timestamp() * 1000)
P2_END   = int(pd.Timestamp("2015-09-19", tz="UTC").timestamp() * 1000)

In [3]:
def main():
    if not INPUT_FILE.exists():
        raise FileNotFoundError(
            f"File sumber tidak ditemukan: {INPUT_FILE.resolve()}"
        )

    # Hapus hasil filter sebelumnya agar tidak tercampur/terduplikasi.
    for output_file in [P1_FILE, P2_FILE]:
        if output_file.exists():
            output_file.unlink()

    total_rows = 0
    p1_rows = 0
    p2_rows = 0

    p1_first_write = True
    p2_first_write = True

    for chunk in pd.read_csv(INPUT_FILE, chunksize=CHUNK_SIZE):
        total_rows += len(chunk)

        # Fold training: 1–15 Juli 2015
        p1 = chunk[
            (chunk["timestamp"] >= P1_START) &
            (chunk["timestamp"] < P1_END)
        ]

        if not p1.empty:
            p1.to_csv(
                P1_FILE,
                index=False,
                mode="w" if p1_first_write else "a",
                header=p1_first_write
            )
            p1_first_write = False
            p1_rows += len(p1)

        # Fold testing: 1–18 September 2015
        p2 = chunk[
            (chunk["timestamp"] >= P2_START) &
            (chunk["timestamp"] < P2_END)
        ]

        if not p2.empty:
            p2.to_csv(
                P2_FILE,
                index=False,
                mode="w" if p2_first_write else "a",
                header=p2_first_write
            )
            p2_first_write = False
            p2_rows += len(p2)

        print(
            f"Dibaca: {total_rows:,} | "
            f"P1: {p1_rows:,} | P2: {p2_rows:,}"
        )

    # Jika suatu periode tidak memiliki baris, tetap buat CSV dengan header.
    columns = [
        "timestamp", "visitorid", "event", "itemid", "transactionid"
    ]

    if p1_first_write:
        pd.DataFrame(columns=columns).to_csv(P1_FILE, index=False)

    if p2_first_write:
        pd.DataFrame(columns=columns).to_csv(P2_FILE, index=False)

    print("\n=== FILTER SELESAI ===")
    print(f"Total data sumber : {total_rows:,}")
    print(f"Data P1 training  : {p1_rows:,}")
    print(f"Data P2 testing   : {p2_rows:,}")
    print(f"File P1           : {P1_FILE.resolve()}")
    print(f"File P2           : {P2_FILE.resolve()}")

if __name__ == "__main__":
    main()

Dibaca: 200,000 | P1: 0 | P2: 0
Dibaca: 400,000 | P1: 0 | P2: 0
Dibaca: 600,000 | P1: 3,120 | P2: 0
Dibaca: 800,000 | P1: 6,526 | P2: 0
Dibaca: 1,000,000 | P1: 6,526 | P2: 0
Dibaca: 1,200,000 | P1: 6,526 | P2: 41,191
Dibaca: 1,400,000 | P1: 6,526 | P2: 241,191
Dibaca: 1,600,000 | P1: 6,526 | P2: 303,710
Dibaca: 1,800,000 | P1: 6,526 | P2: 303,710
Dibaca: 2,000,000 | P1: 6,526 | P2: 303,710
Dibaca: 2,200,000 | P1: 147,057 | P2: 303,710
Dibaca: 2,400,000 | P1: 323,778 | P2: 303,710
Dibaca: 2,600,000 | P1: 333,778 | P2: 303,710
Dibaca: 2,756,101 | P1: 333,778 | P2: 303,710

=== FILTER SELESAI ===
Total data sumber : 2,756,101
Data P1 training  : 333,778
Data P2 testing   : 303,710
File P1           : C:\Users\ADVAN\UTS-GSD-Rebriane-24031554039\data\processed\events_P1_training.csv
File P2           : C:\Users\ADVAN\UTS-GSD-Rebriane-24031554039\data\processed\events_P2_testing.csv
